# Smart Environmental Monitoring: Exploratory Data Analysis

This notebook analyzes the cleaned AQI dataset independently from the FastAPI application. It uses only columns present in the dataset and saves figures to `ml/outputs/eda`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Resolve paths from this notebook's location, not from the FastAPI app.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'ml').is_dir():
    PROJECT_ROOT = Path.cwd().parent.parent

DATASET_PATH = PROJECT_ROOT / 'ml' / 'datasets' / 'cleaned_city_day.csv'
OUTPUT_DIR = PROJECT_ROOT / 'ml' / 'outputs' / 'eda'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (11, 6)

df = pd.read_csv(DATASET_PATH)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date', 'AQI']).sort_values('Date').reset_index(drop=True)

pollutants = [
    column for column in ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3',
                           'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']
    if column in df.columns
]
available_columns = df.columns.tolist()
print(f'Dataset shape: {df.shape}')
print(f'Available pollutant columns: {pollutants}')
print(f'Cities: {df["City"].nunique() if "City" in df.columns else 0}')

## AQI distribution

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df['AQI'], bins=40, kde=True, color='#176b87', ax=ax)
ax.set(title='AQI Distribution', xlabel='AQI', ylabel='Records')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'aqi_distribution.png', dpi=150)
plt.show()

## PM2.5 and PM10 distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, column, color in zip(axes, ['PM2.5', 'PM10'], ['#d95f02', '#1b9e77']):
    if column in df.columns:
        sns.histplot(df[column].dropna(), bins=40, kde=True, color=color, ax=ax)
        ax.set(title=f'{column} Distribution', xlabel=column, ylabel='Records')
    else:
        ax.set_visible(False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'pm25_pm10_distributions.png', dpi=150)
plt.show()

## Environmental parameter correlation

In [ ]:
correlation_columns = [column for column in ['AQI'] + pollutants if column in df.columns]
correlation = df[correlation_columns].corr()
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(correlation, cmap='vlag', center=0, annot=True, fmt='.2f',
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Environmental Parameter Correlation')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'environmental_correlation.png', dpi=150)
plt.show()

## AQI trend over time

In [ ]:
daily_aqi = df.groupby('Date', as_index=False)['AQI'].mean()
daily_aqi['AQI_30_day_mean'] = daily_aqi['AQI'].rolling(30, min_periods=1).mean()
fig, ax = plt.subplots(figsize=(14, 6))
sns.lineplot(data=daily_aqi, x='Date', y='AQI', alpha=0.35, label='Daily mean', ax=ax)
sns.lineplot(data=daily_aqi, x='Date', y='AQI_30_day_mean', color='#b2182b',
             label='30-day rolling mean', ax=ax)
ax.set(title='AQI Trend Over Time', xlabel='Date', ylabel='Mean AQI')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'aqi_trend_over_time.png', dpi=150)
plt.show()

## Pollution trends

In [ ]:
monthly_pollution = (
    df.set_index('Date')[pollutants]
      .resample('MS')
      .mean()
      .reset_index()
)
long_pollution = monthly_pollution.melt('Date', var_name='Pollutant', value_name='Mean concentration')
fig, ax = plt.subplots(figsize=(14, 7))
sns.lineplot(data=long_pollution, x='Date', y='Mean concentration', hue='Pollutant', ax=ax)
ax.set(title='Monthly Pollution Trends', xlabel='Date', ylabel='Mean concentration')
ax.legend(title='Pollutant', bbox_to_anchor=(1.02, 1), loc='upper left')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'pollution_trends.png', dpi=150)
plt.show()

## City/location comparison

In [ ]:
if 'City' in df.columns:
    city_summary = (
        df.groupby('City', as_index=False)
          .agg(mean_aqi=('AQI', 'mean'), records=('AQI', 'size'))
          .query('records >= 30')
          .sort_values('mean_aqi', ascending=False)
    )
    comparison = pd.concat([city_summary.head(10), city_summary.tail(10)]).drop_duplicates()
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.barplot(data=comparison, x='mean_aqi', y='City', hue='mean_aqi',
                palette='Spectral', legend=False, ax=ax)
    ax.set(title='City Comparison by Mean AQI', xlabel='Mean AQI', ylabel='City')
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'city_aqi_comparison.png', dpi=150)
    plt.show()
    display(city_summary.head(10))
else:
    print('City comparison skipped: City column is not available.')

## Output summary

The generated figures are written to `ml/outputs/eda`. This notebook is analysis-only and does not import or modify the FastAPI application.

## Feature engineering and regression model comparison

The prediction target is continuous `AQI`, so this is a regression problem. The selected predictors are `PM2.5`, `PM10`, `CO`, `NO2`, `SO2`, and `O3`; `CO2` is not present in the dataset and is excluded. The full regression workflow is implemented in `ml/training/feature_engineering.py` and `ml/training/model_comparison.py`, with MAE, MSE, RMSE, and R² reported for Random Forest, Decision Tree, and Linear Regression.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

candidate_features = ['PM2.5', 'PM10', 'CO', 'CO2', 'NO2', 'SO2', 'O3']
selected_features = [column for column in candidate_features if column in df.columns]
unavailable_features = [column for column in candidate_features if column not in df.columns]
target = 'AQI'
X = df[selected_features].apply(pd.to_numeric, errors='coerce')
y = pd.to_numeric(df[target], errors='coerce')
valid_rows = X.notna().all(axis=1) & y.notna()
X = X.loc[valid_rows]
y = y.loc[valid_rows]

print('Selected features:', selected_features)
print('Unavailable candidate features:', unavailable_features)
print('Target:', target, '(continuous regression target)')
print('Why selected: available pollutant measurements with direct relevance to AQI.')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Linear Regression': LinearRegression(),
}
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, predictions),
        'MSE': mse,
        'RMSE': mse ** 0.5,
        'R2': r2_score(y_test, predictions),
    })

model_results = pd.DataFrame(results).sort_values('R2', ascending=False)
display(model_results.style.format({'MAE': '{:.2f}', 'MSE': '{:.2f}', 'RMSE': '{:.2f}', 'R2': '{:.4f}'}))
model_results.to_csv(OUTPUT_DIR / 'regression_model_comparison.csv', index=False)